# Mood-to-Movie Recommender

Idea: type in how I'm feeling, get a few movies that match the vibe. This is genre-rule matching on top of MovieLens 100K — not a real emotion classifier. I'm reusing the data from project 10 (`../../data/movielens/`), so no fresh download.

The genre tags in u.item are 19 binary columns. I map a handful of emotions onto subsets of those genres, then rank candidate movies by their average rating among users who rated them at least 30 times. A bit hand-wavy as an emotion model but it's about right for a Friday-night picker.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('../../data/movielens')
assert (DATA / 'u.data').exists(), f'expected MovieLens 100K at {DATA.resolve()}'

GENRES = ['unknown','Action','Adventure','Animation','Children','Comedy','Crime',
          'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery',
          'Romance','Sci-Fi','Thriller','War','Western']
item_cols = ['movie_id','title','release_date','video_release','imdb_url'] + GENRES

movies = pd.read_csv(DATA / 'u.item', sep='|', encoding='latin-1',
                     names=item_cols, usecols=range(len(item_cols)))
ratings = pd.read_csv(DATA / 'u.data', sep='\t',
                      names=['user_id','movie_id','rating','timestamp'])
print('movies', movies.shape, 'ratings', ratings.shape)

movies (1682, 24) ratings (100000, 4)


In [2]:
agg = ratings.groupby('movie_id')['rating'].agg(['mean', 'count']).reset_index()
agg.columns = ['movie_id', 'avg_rating', 'n_ratings']
movies = movies.merge(agg, on='movie_id', how='left').fillna({'avg_rating': 0, 'n_ratings': 0})
# 30 is a small floor but the dataset has a long tail of one-shot ratings.
popular = movies[movies['n_ratings'] >= 30].copy()
print('candidate pool after popularity filter:', len(popular))

candidate pool after popularity filter: 806


## Emotion -> genre map

Loose mappings — these are my opinions, not science. 'Sad' is intentionally complicated: sometimes I want catharsis (Drama, Romance) and sometimes a pick-me-up. I went with catharsis here because that's what I usually want.

In [3]:
EMOTION_TO_GENRES = {
    'happy':    ['Comedy', 'Musical', 'Animation', 'Children'],
    'sad':      ['Drama', 'Romance'],
    'angry':    ['Action', 'Thriller', 'War'],
    'scared':   ['Horror', 'Mystery'],
    'romantic': ['Romance', 'Drama'],
    'curious':  ['Documentary', 'Sci-Fi', 'Mystery'],
    'bored':    ['Action', 'Adventure', 'Sci-Fi'],
    'nostalgic':['Drama', 'Romance', 'Animation'],
}
list(EMOTION_TO_GENRES)

['happy',
 'sad',
 'angry',
 'scared',
 'romantic',
 'curious',
 'bored',
 'nostalgic']

In [4]:
def recommend_for_mood(mood, k=5, require_all=False):
    mood = mood.lower().strip()
    if mood not in EMOTION_TO_GENRES:
        raise ValueError(f'unknown mood {mood!r}. try one of {list(EMOTION_TO_GENRES)}')
    gs = EMOTION_TO_GENRES[mood]
    if require_all:
        mask = popular[gs].all(axis=1)
    else:
        mask = popular[gs].any(axis=1)
    pool = popular.loc[mask].copy()
    # Rank by avg rating with a tiny popularity nudge, otherwise the very top is dominated
    # by a few high-rated 30-vote movies.
    pool['score'] = pool['avg_rating'] + 0.1 * np.log1p(pool['n_ratings'])
    return pool.sort_values('score', ascending=False).head(k)[
        ['title', 'avg_rating', 'n_ratings'] + gs
    ]

recommend_for_mood('happy', k=5)

,title,avg_rating,n_ratings,Comedy,Musical,Animation,Children
407,"Close Shave, A (1995)",4.491071,112,1,0,1,0
168,"Wrong Trousers, The (1993)",4.466102,118,1,0,1,0
113,Wallace & Gromit: The Best of Aardman Animatio...,4.447761,67,0,0,1,0
479,North by Northwest (1959),4.284916,179,1,0,0,0
172,"Princess Bride, The (1987)",4.172840,324,1,0,0,0


In [5]:
recommend_for_mood('angry', k=5)

,title,avg_rating,n_ratings,Action,Thriller,War
317,Schindler's List (1993),4.466443,298,0,0,1
482,Casablanca (1942),4.456790,243,0,0,1
49,Star Wars (1977),4.358491,583,1,0,1
407,"Close Shave, A (1995)",4.491071,112,0,1,0
11,"Usual Suspects, The (1995)",4.385768,267,0,1,0


In [6]:
# require_all=True is a stricter intersection. Useful when the mood maps to >1 genre.
recommend_for_mood('romantic', k=5, require_all=True)

,title,avg_rating,n_ratings,Romance,Drama
482,Casablanca (1942),4.456790,243,1,1
312,Titanic (1997),4.245714,350,1,1
171,"Empire Strikes Back, The (1980)",4.204360,367,1,1
169,Cinema Paradiso (1988),4.173554,121,1,1
196,"Graduate, The (1967)",4.104603,239,1,1


## Honest caveats

- This isn't an emotion model — it's a lookup table. A real version would take free-text input, classify it (e.g. with a small distilled model or even a heuristic), and *then* map to genres. I sketched the API that way (`recommend_for_mood`) so swapping in a classifier later is cheap.
- I'm picking from the popular tail because MovieLens 100K is from the '90s. Without the popularity floor the recommender returns obscure titles with two perfect ratings.
- TMDB genres would be more current but require an API call; I'm sticking with the local MovieLens metadata for now.